# Oppgave 3

# BARE SE MEN IKKE RØRE. DET HAR MOR TIL OLE SAGT

Følgene funksjoner er brukt for å generalisere problemstillinene i oppgaven, og gjennbrukes i samtlige oppgaver

In [2]:
from tqdm import trange
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

#konstanter

def V_1(
        x : np.ArrayLike,
        k : float       = 1,
        N_x : float     = 100,
        alpha : float   = 0.8
        ) -> np.ArrayLike : 
    """
    params:
        x:      Arraylike med heltallsposisjoner
        k:      konstant som tilsvarer k * beta
        N_x:    antall binding sites i én periode
        alpha:  konstant
    return:
        pot:    Potentialet V_1
    """
    x_modN =x%N_x

    pot = -k*(x_modN-N_x)/((1 - alpha)*N_x)
    pot[x_modN <= alpha * N_x]  =(k * x_modN/(alpha * N_x))[x_modN <= alpha * N_x]

    return pot
    
def V_2(x, *args) -> int:
    return np.zeros_like(x)

def probabilities(
        V : function,
        x_0 : np.ArrayLike,
        k : float,
        N_x : float,
        alpha : float
        ) -> np.ArrayLike : 
    """
    params:
        V:    potensialfunksjonen sannsynlighetene beregnes ut ifra
        x_0:      ArrayLike med posisjoner, hvis sannsynlighet for forflyttning skal finnes
        *args:  parameterene som potensialet tar inn

    return:
        
    """

    pot_minus = V(x_0 - 1, k, N_x, alpha)
    pot_zero  = V(x_0, k, N_x, alpha)
    pot_plus  = V(x_0 + 1, k, N_x, alpha)

    p_minus = (1 + np.exp(pot_minus - pot_plus) + np.exp(pot_minus - pot_zero))**-1
    #p_zero = (1 + np.exp(pot_zero - pot_plus) + np.exp(pot_zero - pot_minus))**-1
    p_plus = (1 + np.exp(pot_plus - pot_zero) + np.exp(pot_plus - pot_minus))**-1
    
    return p_minus, p_plus

def simulate3(
        alpha : float  = 0.8, 
        T_p : int = 500, 
        N_x : int = 100, 
        N_p : int = 1200,
        k : int  = 1000,
        spatialPeriods : int  = 2,
        cycles : int = 10):
    
    N = N_x * spatialPeriods 
    XX = []                                                   #number of binding sites
    x_ref : np.ndarray  = np.arange(N)                                          #reference array of positions
    x_0 : np.ndarray = np.asarray([i for i in range(N) for _ in range(N_p//N)]) #array of all particles
    XX.append(x_0)

    p_1 = probabilities(V_1, x_ref, k, N_x, alpha)
    p_2 = probabilities(V_2, x_ref, k, N_x , alpha)

    pm_tup = (p_1[0], p_2[0])
    pp_tup = (p_1[1], p_2[1])

    n = np.zeros_like(x_0)
    currents = np.zeros(cycles)

    for c in trange(2 * cycles, ascii = True, desc  = "runnning half-cycles"):
        pm  = pm_tup[c%2]
        pp =  pp_tup[c%2]

        for t in range(T_p):
            x = x_0%N
            prob_minus = np.asarray([pm[i] for i in x])
            prob_plus = np.asarray([pp[i] for i in x])

            r = np.random.rand(N_p)
            x[r < prob_minus] -= 1
            x[1 - prob_plus < r] += 1

            n += x - x_0%N
            x_0 = x

            XX.append(x_0)
            
        if c%2:
            currents[c//2] = np.mean(n)/(2 * T_p)
            n = np.zeros_like(x)

    return x, currents, XX

### 3a

simulering med $\alpha = 0.8$.

In [ ]:
x, currents, XX = simulate3(alpha = 0.8)

for i in range(len(currents)):
    print(f"{i}: {currents[i]:.6f}")

runnning half-cycles: 100%|##########| 20/20 [00:04<00:00,  4.59it/s]

0: 0.000000
1: 1.000000
2: 2.000000
3: 3.000000
4: 4.000000
5: 5.000000
6: 6.000000
7: 7.000000
8: 8.000000
9: 9.000000


Man får netto strøm mot venstre ettersom partiklene samles opp i minimumspunktene til $V_1$, og deretter blir eksitert til tilstand $V_2$, hvor de diffunderer. Ettersom nærmeste lokale maksimum for partiklene er det til venstre for der de ble samlet opp, er det flere partikler som forflytter seg forbi dette, enn det er partikler som forflytter seg forbi nærmeste lokale maksimum mot høyre. Når de deeksiteres vil disse partiklene samles opp i et annet lokalt minimum, noe som fører til netto strøm.

Dette forklarer hvorfor strømmen er større til å start med, ettersom de da var uniformt fordelt, og ikke fordelt rundt minimumspunktene slik som i resten av syklene. 

In [4]:
x, currents, XX = simulate3(alpha = 0.1)

for i in range(len(currents)):
    print(f"{i}: {currents[i]:.6f}")

runnning half-cycles: 100%|##########| 20/20 [00:04<00:00,  4.53it/s]

0: 0.039632
1: 0.028069
2: 0.027128
3: 0.028081
4: 0.030120
5: 0.028403
6: 0.028681
7: 0.030189
8: 0.028809
9: 0.028468


For $\alpha = 0.1$ har vi en tilsvarende situasjon, men hvor lokale mimium ligger rett til venstre for lokale maksimum, og man får dermed strøm mot høyre. Det ligger nærmere enn for $\alpha = 0.8$ så derfor er strømmen større.

### 3b

In [5]:
T_p_arr = np.linspace(1, 1001, 50)

for tp in T_p_arr:
    x, currents, XX = simulate3(
        T_p = tp,
        cycles = 1,
        N_p = 4000)


runnning half-cycles:   0%|          | 0/2 [00:00<?, ?it/s]


TypeError: 'numpy.float64' object cannot be interpreted as an integer